In [6]:
from custom.tools import h5set
import os
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from tqdm import tqdm

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
else: device = torch.device('cpu')
print(f'Selected device: {device}')

Selected device: cuda


In [2]:
dir_path = "/mnt/data/train_test_val"
h5_path = "../data/H1_rechunked.h5"
dir_ = os.listdir(dir_path)
if len(dir_)==0:
    noise = h5set(path=h5_path, dataset='noise')
    train, val, test = torch.utils.data.random_split(noise, [225000, 75000, 25000])
    injection = h5set(path=h5_path, dataset='injection')
    test = ConcatDataset([test, injection])
    torch.save(train, "/mnt/data/train_test_val/train.pt") #just saves indices
    torch.save(val, "/mnt/data/train_test_val/val.pt")
    torch.save(test, "/mnt/data/train_test_val/test.pt")
else:
    train = torch.load("/mnt/data/train_test_val/train.pt", weights_only=False)
    val = torch.load("/mnt/data/train_test_val/val.pt", weights_only=False)
    test = torch.load("/mnt/data/train_test_val/test.pt", weights_only=False)


training = DataLoader(train,
                batch_size=1,
                shuffle=True,
                num_workers=0,
                #persistent_workers=True,
                drop_last=True,
                )


In [2]:
# local run
data = h5set('l1500mb_c.h5', 100, 30)
training = DataLoader(data, batch_size=1, num_workers=os.cpu_count())

In [59]:
AE = AEric(num_feat=1,
            exp_dim=32,
            compr_dim=8,
            num_layers=3,
            v=True)
AE.to(device)
loss_func = torch.nn.MSELoss()
lr = 5e-3
optim = torch.optim.Adam(AE.parameters(),
                        lr=lr,
                        weight_decay=1e-5
                        )



In [3]:
def traingio(model, device, dataloader, loss_fn, optim, scaler, clip):
    model.train()
    epoch_loss = 0
    for batch_data in tqdm(dataloader):
        optim.zero_grad()
        # print(batch_data.element_size() * batch_data.nelement())
        batch_data = batch_data.reshape(-1, 100, 1).to(device)
        # print(batch_data.shape)
        with torch.autocast(str(device), dtype=torch.float16):
            output = model(batch_data)
            loss = loss_fn(output, batch_data)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        if clip > 0: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(optim)
        scaler.update()
        # loss = np.sqrt(loss.item()) # if need
        epoch_loss += loss
    return epoch_loss / len(dataloader)

In [5]:
def train_epoch(ae, device, dataloader,timestep, loss_fn, optim):
    ae.train()
    losses = []
    for batch_data in dataloader:
        num_time_steps = batch_data.shape[1]
        remainder = num_time_steps % timestep
        batch_data = batch_data[:, :-remainder]
        for sample_idx in range(batch_data.shape[0]):
            sequence = batch_data[sample_idx, :]
            segments = sequence.reshape(-1, timestep, 1)
            for segment_idx in range(segments.shape[0]):
                current_window = segments[segment_idx, :, :]
                c_w = current_window.unsqueeze(0)
                c_w = c_w.to(device)
                ae_output = ae(c_w)
                loss = loss_fn(ae_output,c_w)
                print(loss)
                optim.zero_grad()
                loss.backward()
                optim.step()

                losses.append(loss.detach().cpu().numpy())
    losses = np.mean(losses)
    return losses

In [20]:
num_epochs = 10
losses = []
for epoch in range(num_epochs):
    ### Training (use the training function)
    train_loss = traingio(
        model=AE,
        device=device,
        dataloader=training,
        loss_fn=loss_func,
        optim=optim)
    print(f'TRAIN - EPOCH {epoch+1}/{num_epochs} - loss: {train_loss}')
    losses.append(train_loss)

NameError: name 'AE' is not defined

In [4]:
class EricGio(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.El1 = nn.LSTM(input_size=1, hidden_size=32, num_layers=3, batch_first=True)  # Send whole output (corresponds to 100x32 in paper)
  
        self.El2 = nn.LSTM(input_size=32, hidden_size=8, num_layers=3, batch_first=True)  # Send last output (corresponds to whole[-1], has 1 value for each layer (3)) I guess the paper takes only the last of these x[-1]
        
        self.Dl1 = nn.LSTM(input_size=8, hidden_size=8, num_layers=3, batch_first=True)
        
        self.Dl2 = nn.LSTM(input_size=8, hidden_size=32, num_layers=3, batch_first=True)
        
        self.TimeDistributed = nn.Conv1d(in_channels=32, out_channels=1, kernel_size=1)   # Not sure it corresponds to TimeDistributed(Dense), also stoopid LSTM output are weird, and does not work with Conv1d. Perhaps we ought to do manual labor

    def forward(self, x):
        ## Encoding
        # print(f'Input {x.shape=}')
        x, _ = self.El1(x)  
        # print(f"First LSTM out {x.shape=}")
        _, (x, _) = self.El2(x)  
        # print(f"Second LSTM out {x[-1].shape=}")

        ## Repeating
        x = x[-1].unsqueeze(1).repeat(1, 100, 1)  # Tensor.unsqueeze(x) adds a dimension to x position, to have batch dim back.  ## gio method

        ## Decoding
        # print("Decoding")
        x, _ = self.Dl1(x)
        # print(f"First LSTM out {x.shape=}")
        x, _ = self.Dl2(x)
        # print(f"Second LSTM out {x.shape=}")
        x = torch.movedim(x, 1, 2)
        # print(f"3D transposed {x.shape=}")
        x = self.TimeDistributed(x)
        # print(f'Convoluted {x.shape=}')
        x = torch.movedim(x, 1, 2)
        # print(f'Back to original dim {x.shape=}')
        return x

In [ ]:
ehi = EricGio()
ehi.to(device)
ehi = torch.compile(ehi)
loss_fn = nn.MSELoss()
optim = torch.optim.Adam(ehi.parameters())
scaler = torch.amp.GradScaler()
for _ in range(2):
    traingio(ehi, device, training, loss_fn, optim, scaler, 0.2)

In [8]:
import custom.instructions as ist

ist.evalgio(ehi, device, val)

In [11]:
ehi = EricGio()
ehi.to(device)
with torch.no_grad():
    ehi.eval()
    ehi(torch.randn(543, 100, 1).to(device))

Input x.shape=torch.Size([543, 100, 1])
First LSTM out x.shape=torch.Size([543, 100, 32])
Second LSTM out x[-1].shape=torch.Size([543, 8])
Repeated x.shape=torch.Size([543, 100, 8])
Decoding
First LSTM out x.shape=torch.Size([543, 100, 8])
Second LSTM out x.shape=torch.Size([543, 100, 32])
3D transposed x.shape=torch.Size([543, 32, 100])
Convoluted x.shape=torch.Size([543, 1, 100])
Back to original dim x.shape=torch.Size([543, 100, 1])


In [93]:
ehi = EricGio()
ehi.to(device)
with torch.no_grad():
    ehi.eval()
    ehi(torch.randn(543, 100, 1).to(device))

Input x.shape=torch.Size([543, 100, 1])
First LSTM out x.shape=torch.Size([543, 100, 32])
Second LSTM out x[:,-1,:].shape=torch.Size([543, 8])
Repeated x.shape=torch.Size([1, 54300, 8])
Decoding
First LSTM out x.shape=torch.Size([1, 54300, 8])
Second LSTM out x.shape=torch.Size([1, 54300, 32])
3D transposed x.shape=torch.Size([1, 32, 54300])
Convoluted x.shape=torch.Size([1, 1, 54300])
Back to original dim x.shape=torch.Size([1, 54300, 1])
